# FastPower (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_java.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `FastPower`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono campi `static` della classe, e la traccia si attiva con `trace = true`.

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Elevamento a potenza veloce: FastPower

**Input:** un numero $x$ e un numero naturale $n$. **Output:** $x^n$, per quadrati ripetuti. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor + 2$ chiamate, al più $2(\lfloor\log_2 n\rfloor + 1)$ moltiplicazioni.

Pseudocodice: sezione 14.7 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
%%writefile FastPower.java
public class FastPower
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static int floorLog2(int n)                    // floor(log2 n) con un ciclo (solo per la formula dei passi)
    {
        int k = 0;
        while (n > 1)
        {
            n = n / 2;
            k = k + 1;
        }
        return k;
    }

    // FastPower(x, n)   i valori troppo grandi traboccano in C++ e Java, non in Python
    static long fastPower(long x, int n)
    {
        push(n); cnt[2]++;
        if (n == 0)
        {
            cnt[3]++;
            if (trace) say(String.format("FastPower(%d, 0)", x), "caso base, restituisce 1");
            pop();
            return 1;
        }
        if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("%d != 0 => chiama FastPower(%d, %d)   (%d/2 = %d)", n, x, n / 2, n, n / 2));
        long y = fastPower(x, n / 2); cnt[5]++;    // una sola chiamata ricorsiva: il risultato e' in y
        cnt[6]++;
        long r;
        if (n % 2 == 0)
        {
            cnt[7]++;
            mults += 1;
            r = y * y;
            if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("y = %d, %d e' pari => restituisce %d * %d = %d", y, n, y, y, r));
        }
        else
        {
            cnt[9]++;
            mults += 2;
            r = x * y * y;
            if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("y = %d, %d e' dispari => restituisce %d * %d * %d = %d", y, n, x, y, y, r));
        }
        pop();
        return r;
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: FastPower(3, 13)
        long x = 3;
        int n = 13;
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        long r = fastPower(x, n);
        trace = false;
        System.out.printf("%d^%d = %d, chiamate = %d, moltiplicazioni = %d\n", x, n, r, calls, mults);
        printSteps(2, 3, 5, 6, 7, 9);
        System.out.printf("S(n) = 4 floor(log2 n) + 6 = %d\n", 4 * floorLog2(n) + 6);

        // Quante moltiplicazioni: circa 2 log2 n invece di n (x = 1, per non traboccare)
        System.out.printf("\n");
        System.out.printf("      n | chiamate | moltiplicazioni | S(n) | 4 floor(log2 n) + 6\n");
        int[] values = {1, 2, 13, 100, 1000, 1000000};
        for (int v : values)
        {
            reset();
            fastPower(1, v);
            System.out.printf("%7d | %8d | %15d | %4d | %d\n", v, calls, mults, total(), 4 * floorLog2(v) + 6);
        }
    }
}

Writing FastPower.java


In [3]:
!javac FastPower.java && java FastPower

chiamata                    | azione
FastPower(3, 13)            | 13 != 0 => chiama FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => chiama FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => chiama FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => chiama FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | caso base, restituisce 1
      FastPower(3, 1)       | y = 1, 1 e' dispari => restituisce 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 e' dispari => restituisce 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 e' pari => restituisce 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 e' dispari => restituisce 3 * 729 * 729 = 1594323
3^13 = 1594323, chiamate = 5, moltiplicazioni = 7
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Passi totali S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n | chiamate | moltiplicazioni | S(n) | 4 f

**Prova tu.** Cambia $x$ e $n$ e conta le moltiplicazioni.

In [4]:
%%writefile ProvaFastPower.java
public class ProvaFastPower
{
    public static void main(String[] args)
    {
        long x = 2;
        int n = 10;        // <-- cambia x e n (con x e n grandi C++ e Java traboccano)
        FastPower.reset();
        FastPower.trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        long r = FastPower.fastPower(x, n);
        FastPower.trace = false;
        System.out.printf("%d^%d = %d, moltiplicazioni = %d\n", x, n, r, FastPower.mults);
    }
}

Writing ProvaFastPower.java


In [5]:
!javac ProvaFastPower.java && java ProvaFastPower

chiamata                    | azione
FastPower(2, 10)            | 10 != 0 => chiama FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => chiama FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => chiama FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => chiama FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | caso base, restituisce 1
      FastPower(2, 1)       | y = 1, 1 e' dispari => restituisce 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 e' pari => restituisce 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 e' dispari => restituisce 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 e' pari => restituisce 32 * 32 = 1024
2^10 = 1024, moltiplicazioni = 6


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).